# 02 — Feature Engineering
**TF-IDF Vectorization + BERT Embeddings**

This notebook:

1. Builds TF-IDF feature matrices (fit on train, transform test)
2. Generates BERT sentence embeddings via `sentence-transformers`
3. Saves all matrices + the fitted vectorizer / model to `../data/features/`

## Imports & Config

In [ ]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
from tqdm.auto import tqdm

# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# BERT embeddings
from sentence_transformers import SentenceTransformer

In [ ]:
import os
from google.colab import drive



drive.mount('/content/drive')
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/SmartReviewAnalyzer'




#Paths
PROCESSED_DIR = os.path.join(DRIVE_PROJECT_ROOT, 'data/processed')
FEATURES_DIR  = os.path.join(DRIVE_PROJECT_ROOT, 'data/features')

TRAIN_CSV = os.path.join(PROCESSED_DIR, 'preprocessed_train.csv')
TEST_CSV  = os.path.join(PROCESSED_DIR, 'preprocessed_test.csv')

os.makedirs(FEATURES_DIR, exist_ok=True)

#TF-IDF hyper-parameters
TFIDF_MAX_FEATURES   = 50_000
TFIDF_NGRAM_RANGE    = (1, 2)
TFIDF_MIN_DF         = 2
TFIDF_SUBLINEAR_TF   = True

#BERT model
BERT_MODEL_NAME = 'all-MiniLM-L6-v2'
BERT_BATCH_SIZE = 64

print('Config ready (Google Drive Connected).')
print(f'  Features dir : {FEATURES_DIR}')
print(f'  TF-IDF vocab : {TFIDF_MAX_FEATURES:,} max features')
print(f'  BERT model   : {BERT_MODEL_NAME}')

## Load Preprocessed Data

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f'Train : {df_train.shape}')
print(f'Test  : {df_test.shape}')
df_train.head(2)

In [ ]:
# Separate text columns and labels
train_text_tfidf = df_train['cleaned_text_tfidf'].fillna('').tolist()
test_text_tfidf  = df_test['cleaned_text_tfidf'].fillna('').tolist()

train_text_bert  = df_train['cleaned_text_bert'].fillna('').tolist()
test_text_bert   = df_test['cleaned_text_bert'].fillna('').tolist()

y_train = df_train['binary_label'].values
y_test  = df_test['binary_label'].values

print(f'Train texts (TF-IDF) : {len(train_text_tfidf):,}')
print(f'Test  texts (TF-IDF) : {len(test_text_tfidf):,}')
print(f'Train texts (BERT)   : {len(train_text_bert):,}')
print(f'Test  texts (BERT)   : {len(test_text_bert):,}')

---
## TF-IDF Vectorization

### Initialize TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(
    max_features = TFIDF_MAX_FEATURES,
    ngram_range  = TFIDF_NGRAM_RANGE,
    min_df       = TFIDF_MIN_DF,
    sublinear_tf = TFIDF_SUBLINEAR_TF,
    strip_accents= 'unicode',
    analyzer     = 'word',
    token_pattern= r'\b[a-zA-Z][a-zA-Z]+\b',
)

print('TfidfVectorizer initialised.')
print(f'  max_features  : {TFIDF_MAX_FEATURES:,}')
print(f'  ngram_range   : {TFIDF_NGRAM_RANGE}')
print(f'  min_df        : {TFIDF_MIN_DF}')
print(f'  sublinear_tf  : {TFIDF_SUBLINEAR_TF}')

### Fit on Training Data

In [ ]:
print('Fitting TF-IDF on training corpus ...')
X_train_tfidf = tfidf.fit_transform(train_text_tfidf)

print(f'Vocabulary size  : {len(tfidf.vocabulary_):,}')
print(f'Train matrix     : {X_train_tfidf.shape}  (sparse float32)')
print(f'Non-zero entries : {X_train_tfidf.nnz:,}')

### Transform Validation / Test Data

In [ ]:
print('Transforming test corpus ...')
X_test_tfidf = tfidf.transform(test_text_tfidf)

print(f'Test matrix : {X_test_tfidf.shape}  (sparse float32)')

# Sanity check — top tokens by mean TF-IDF weight on training set
feature_names = np.array(tfidf.get_feature_names_out())
mean_scores   = np.asarray(X_train_tfidf.mean(axis=0)).ravel()
top10_idx     = mean_scores.argsort()[::-1][:10]

print('\nTop-10 features by mean TF-IDF weight (train):')
for rank, idx in enumerate(top10_idx, 1):
    print(f'  {rank:2d}. {feature_names[idx]:<25s}  {mean_scores[idx]:.5f}')

### Save TF-IDF Features

In [ ]:
# Sparse matrices → .npz (preserves sparsity, fast I/O)
tfidf_train_path = os.path.join(FEATURES_DIR, 'X_train_tfidf.npz')
tfidf_test_path  = os.path.join(FEATURES_DIR, 'X_test_tfidf.npz')
tfidf_model_path = os.path.join(FEATURES_DIR, 'tfidf_vectorizer.joblib')
labels_path      = os.path.join(FEATURES_DIR, 'labels.npz')

sp.save_npz(tfidf_train_path, X_train_tfidf)
sp.save_npz(tfidf_test_path,  X_test_tfidf)
joblib.dump(tfidf, tfidf_model_path)
np.savez(labels_path, y_train=y_train, y_test=y_test)

print('TF-IDF artefacts saved:')
for p in [tfidf_train_path, tfidf_test_path, tfidf_model_path, labels_path]:
    size_mb = os.path.getsize(p) / 1_048_576
    print(f'  {p}  ({size_mb:.2f} MB)')

---
## BERT Embeddings (Option B — Recommended)

Using [`sentence-transformers`](https://www.sbert.net/) with **`all-MiniLM-L6-v2`**:
- 6-layer MiniLM distilled from BERT — ~80 MB download
- **384-dimensional** embeddings, L2-normalised
- Excellent quality/speed tradeoff for sentiment classification


### Load BERT Model

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

print(f'Loading {BERT_MODEL_NAME} ...')
bert_model = SentenceTransformer(BERT_MODEL_NAME, device=device)

embedding_dim = bert_model.get_sentence_embedding_dimension()
print(f'Model loaded  —  embedding dim : {embedding_dim}')

### Convert Text → Embeddings

In [ ]:
print(f'Encoding {len(train_text_bert):,} training examples ...')
X_train_bert = bert_model.encode(
    train_text_bert,
    batch_size           = BERT_BATCH_SIZE,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True,
)

print(f'Train embedding matrix : {X_train_bert.shape}  dtype={X_train_bert.dtype}')

In [ ]:
print(f'Encoding {len(test_text_bert):,} test examples ...')
X_test_bert = bert_model.encode(
    test_text_bert,
    batch_size           = BERT_BATCH_SIZE,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True,
)

print(f'Test  embedding matrix : {X_test_bert.shape}  dtype={X_test_bert.dtype}')

# Sanity — L2 norms should be ~1.0 after normalisation
norms = np.linalg.norm(X_train_bert[:100], axis=1)
print(f'\nEmbedding L2 norms (first 100 train rows):')
print(f'  mean={norms.mean():.4f}  min={norms.min():.4f}  max={norms.max():.4f}')

### Save Embedding Features

In [ ]:
MY_FEATURES_DIR = '/content/drive/MyDrive/SmartReviewAnalyzer_features'
os.makedirs(MY_FEATURES_DIR, exist_ok=True)
# BERT
np.save(f'{MY_FEATURES_DIR}/X_train_bert.npy', X_train_bert)
np.save(f'{MY_FEATURES_DIR}/X_test_bert.npy',  X_test_bert)
with open(f'{MY_FEATURES_DIR}/bert_model_name.txt', 'w') as f:
    f.write(BERT_MODEL_NAME)

